In [ ]:
# perhitungan
import numpy as np
import random

# display
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    "fish_color",
    [
        (0.0,  "#55e2e9"),  # biru muda
        (1.0,  "#ef233c"),  # merah
    ]
)

# animasi
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
EMPTY = 0
FISH = 1

# Inisialisasi grid berukuran gridx x gridy dengan numFish ikan
# Menyimpan informasi di dua grid berbeda
def initializeGrid(gridx, gridy, numFish=20):
  # INPUT: gridx (int), gridy (int), numFish (int)
  # OUTPUT: grid state (gridx x gridy matrix), grid direction (gridx x gridy matrix)
  state = np.zeros((gridy, gridx), dtype=int)
  direction = np.zeros((gridy, gridx), dtype=float)

  placed = 0
  while placed < numFish:
      x = random.randint(0, gridx-1)
      y = random.randint(0, gridy-1)
      if state[y, x] == EMPTY:
          state[y, x] = FISH
          direction[y, x] = random.uniform(0, 2*np.pi)
          placed += 1

  return state, direction

# Jarak periodik antara sel (x1, y1) dengan (x2, y2)
def dist(x1, y1, x2, y2):
  # INPUT: x1 (int), y1 (int), x2 (int), y2 (int)
  # OUTPUT: dist
  dx = min(abs(x1-x2), GRIDX-abs(x1-x2))
  dy = min(abs(y1-y2), GRIDY-abs(y1-y2))
  return max(dx, dy)

# Putaran yang dibutuhkan agar dari menghadap dir jadi menunjuk ke arah (dx, dy)
def turn(dir, dx, dy):
  # INPUT: dir (float), dx (float), dy (float)
  # OUTUT: turnAngle (float)
  if dx == 0 and dy == 0:
      return 0.0
  target = np.arctan2(dy, dx)
  return np.arctan2(np.sin(target-dir), np.cos(target-dir))

# Aturan Pemisahan
PERSONAL_RANGE = 2 # jarak menghindar
def separation(x, y, state, direction, minDist=PERSONAL_RANGE):
  # INPUT: x (int), y (int), state (matrix), direction (matrix), minDist (float)
  # OUTPUT: dx, dy
  dx = 0
  dy = 0
  for j in range(-minDist, minDist+1):
      for i in range(-minDist, minDist+1):
          if i == 0 and j == 0:
              continue
          nx = (x + i) % GRIDX
          ny = (y + j) % GRIDY
          if state[ny, nx] == FISH:
              dx += x - nx
              dy += y - ny
  return dx, dy

# Aturan Penyelarasan
VISION_RANGE = 5 # jarak penglihatan ikan
def alignment(x, y, state, direction, visionDist=VISION_RANGE):
  # INPUT: x (int), y (int), state (matrix), direction (matrix), visionDist (float)
  # OUTPUT: dx, dy
  dx = 0
  dy = 0
  for j in range(-visionDist, visionDist+1):
      for i in range(-visionDist, visionDist+1):
          if i == 0 and j == 0:
              continue
          nx = (x + i) % GRIDX
          ny = (y + j) % GRIDY
          if state[ny, nx] == FISH:
              dx += np.cos(direction[ny, nx])
              dy += np.sin(direction[ny, nx])
  return dx, dy

# Aturan Kohesi
def cohesion(x, y, state, direction, visionDist=VISION_RANGE):
  # INPUT: x (int), y (int), state (matrix), direction (matrix), visionDist (float)
  # OUTPUT: dx, dy
  cx = 0
  cy = 0
  count = 0
  for j in range(-visionDist, visionDist+1):
      for i in range(-visionDist, visionDist+1):
          if i == 0 and j == 0:
              continue
          nx = (x + i) % GRIDX
          ny = (y + j) % GRIDY
          if state[ny, nx] == FISH:
              cx += nx
              cy += ny
              count += 1
  if count == 0:
      return 0.0, 0.0
  cx = cx / count
  cy = cy / count
  return cx - x, cy - y

# Menggabungkan setiap aturan jadi satu fungsi
def updateDirection(x, y, state, direction, sepF=0.4, alignF=0.3, cohF=0.1):
  # INPUT: x (int), y (int), state (matrix), direction (matrix), sepF (float), alignF (float), cohF (float)
  # OUTPUT: dx, dy
  dir0 = direction[y, x]

  sdx, sdy = separation(x, y, state, direction)
  adx, ady = alignment(x, y, state, direction)
  cdx, cdy = cohesion(x, y, state, direction)

  newDir = dir0
  newDir += turn(dir0, sdx, sdy) * sepF # ada faktor yang menentukan signifikansi
  newDir += turn(dir0, adx, ady) * alignF
  newDir += turn(dir0, cdx, cdy) * cohF

  return newDir

# Menggerakan ikan sesuai direction (yang dibulatkan ke salah satu dari arah 1-8)
def step(state, direction):
    newState = np.zeros_like(state)
    newDir = np.zeros_like(direction)

    for y in range(GRIDY):
        for x in range(GRIDX):
            if state[y, x] == FISH:
                dir1 = updateDirection(x, y, state, direction)
                dx = round(np.cos(dir1))
                dy = round(np.sin(dir1))

                nx = (x + dx) % GRIDX # periodic
                ny = (y + dy) % GRIDY
                if newState[ny, nx] == EMPTY:
                    newState[ny, nx] = FISH
                    newDir[ny, nx] = dir1
                else:
                    newState[y, x] = FISH
                    newDir[y, x] = dir1
    return newState, newDir

In [ ]:
GRIDX = 150
GRIDY = 100
NUMFISH = 120

state, direction = initializeGrid(GRIDX, GRIDY, NUMFISH)
fig, ax = plt.subplots(figsize=(6, 4))

def update(frame):
    global state, direction
    state, direction = step(state, direction)

    ax.clear()
    sns.heatmap(state, cbar=False, ax=ax, square=True,
                xticklabels=False, yticklabels=False,
                vmin=0, vmax=1, cmap=cmap)
    ax.set_title(f"Frame {frame}")

anim = FuncAnimation(fig, update, frames=100, interval=80, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())